# 03 — Customer Profitability & Revenue Leakage Analysis

In [2]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

CLEAN = ROOT / "data" / "processed" / "superstore_cleaned.csv"
RAW = ROOT / "data" / "raw" / "Sample - Superstore.csv"

FILE = CLEAN if CLEAN.exists() else RAW

OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

# Print paths to help diagnose problems
print("Project root:", ROOT)
print("Cleaned file exists:", CLEAN.exists())
print("Raw file exists:", RAW.exists())
print("Selected file:", FILE)

if not FILE.exists():
    raise FileNotFoundError(
        "CSV not found. Check that Sample - Superstore.csv "
        "is inside the project's data/raw folder."
    )

for enc in ("utf-8-sig", "cp1252", "latin1"):
    try:
        df = pd.read_csv(FILE, encoding=enc)
        print("Loaded successfully using:", enc)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("Could not decode the CSV file.")

df.columns = [str(c).strip() for c in df.columns]

for col in ["Sales", "Profit", "Discount", "Quantity"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Input:", FILE)
print(f"Rows: {len(df):,}")
display(df.head())


Project root: A:\Data Analyst\Customer-profitability-intellegence
Cleaned file exists: True
Raw file exists: True
Selected file: A:\Data Analyst\Customer-profitability-intellegence\data\processed\superstore_cleaned.csv
Loaded successfully using: utf-8-sig
Input: A:\Data Analyst\Customer-profitability-intellegence\data\processed\superstore_cleaned.csv
Rows: 9,994


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,NaN,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df["Row Profit Margin %"] = np.where(
    df["Sales"].notna() & (df["Sales"] != 0),
    df["Profit"] / df["Sales"] * 100, np.nan
)
df["Loss-making Row"] = df["Profit"] < 0
display(df[["Sales","Profit","Row Profit Margin %","Loss-making Row"]].head())


,Sales,Profit,Row Profit Margin %,Loss-making Row
0,261.9600,41.9136,16.00,False
1,731.9400,219.5820,30.00,False
2,14.6200,6.8714,47.00,False
3,957.5775,-383.0310,-40.00,True
4,22.3680,2.5164,11.25,False


In [4]:
if "Customer Name" in df.columns:
    order_agg = ("Order ID", "nunique") if "Order ID" in df.columns else ("Profit", "size")
    customer = df.groupby("Customer Name", dropna=False).agg(
        Orders=order_agg, Sales=("Sales","sum"), Profit=("Profit","sum"),
        Loss_Making_Rows=("Loss-making Row","sum")
    ).reset_index()
    customer["Profit Margin %"] = np.where(customer["Sales"] != 0, customer["Profit"] / customer["Sales"] * 100, np.nan)
    customer = customer.sort_values("Profit", ascending=False)
    customer.to_csv(OUT / "customer_profitability_summary.csv", index=False)
    display(customer.head(10))
    display(customer.tail(10))


,Customer Name,Orders,Sales,Profit,Loss_Making_Rows,Profit Margin %
730,Tamara Chand,5,19052.218,8981.3239,1,47.140569
622,Raymond Buch,6,15117.339,6976.0959,1,46.146322
671,Sanjit Chand,9,14142.334,5757.4119,2,40.710479
334,Hunter Lopez,6,12873.298,5622.4292,0,43.675127
6,Adrian Barton,10,14473.571,5444.8055,7,37.618950
757,Tom Ashbrook,4,14595.620,4703.7883,1,32.227396
157,Christopher Martinez,4,8954.020,3899.8904,1,43.554631
431,Keith Dawkins,12,8181.256,3038.6254,2,37.141307
35,Andy Reiter,6,6608.448,2884.6208,0,43.650503
194,Daniel Raglin,8,8350.868,2869.0760,4,34.356620


,Customer Name,Orders,Sales,Profit,Loss_Making_Rows,Profit Margin %
557,Natalie Fritzler,7,8322.826,-1695.9714,5,-20.377350
154,Christine Phan,8,5888.275,-1850.3029,7,-31.423514
686,Sean Miller,5,25043.050,-1980.7393,5,-7.909337
684,Sean Braxton,7,8057.891,-2082.7451,6,-25.847273
559,Nathan Cano,6,2218.990,-2204.8072,3,-99.360844
329,Henry Goldwyn,12,3247.642,-2797.9635,5,-86.153692
693,Sharelle Roach,5,3233.481,-3333.9144,5,-103.106046
477,Luke Foster,7,3930.509,-3583.9770,7,-91.183534
307,Grant Thornton,3,9351.212,-4108.6589,2,-43.937181
165,Cindy Stewart,6,5690.055,-6626.3895,2,-116.455632


In [5]:
for col, filename in [
    ("Product Name","product_profitability_summary.csv"),
    ("Sub-Category","subcategory_profitability_summary.csv"),
    ("Region","region_profitability_summary.csv"),
    ("Category","category_profitability_summary.csv")
]:
    if col in df.columns:
        summary = df.groupby(col, dropna=False).agg(Sales=("Sales","sum"), Profit=("Profit","sum"), Rows=("Profit","size")).reset_index()
        summary["Profit Margin %"] = np.where(summary["Sales"] != 0, summary["Profit"] / summary["Sales"] * 100, np.nan)
        summary.sort_values("Profit").to_csv(OUT / filename, index=False)
        print("Lowest-profit entries:", col)
        display(summary.sort_values("Profit").head(10))


Lowest-profit entries: Product Name


,Product Name,Sales,Profit,Rows,Profit Margin %
475,Cubify CubeX 3D Printer Double Head Print,11099.963,-8879.9704,3,-80.000000
985,Lexmark MX611dhe Monochrome Laser Printer,16829.901,-4589.9730,4,-27.272727
476,Cubify CubeX 3D Printer Triple Head Print,7999.980,-3839.9904,1,-48.000000
425,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.640,-2876.1156,5,-29.000000
376,Bush Advantage Collection Racetrack Conference...,9544.725,-1934.3976,7,-20.266667
683,GBC DocuBind P400 Electric Binding System,17965.068,-1878.1662,6,-10.454545
444,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784,1,-8.000000
1043,Martin Yale Chadless Opener Electric Letter Op...,16656.200,-1299.1836,6,-7.800000
285,Balt Solid Wood Round Tables,6518.754,-1201.0581,4,-18.424658
364,BoxOffice By Design Rectangular and Half-Moon ...,1706.250,-1148.4375,3,-67.307692


Lowest-profit entries: Sub-Category


,Sub-Category,Sales,Profit,Rows,Profit Margin %
16,Tables,206965.5320,-17725.4811,319,-8.564460
4,Bookcases,114879.9963,-3472.5560,228,-3.022768
15,Supplies,46673.5380,-1189.0995,190,-2.547695
8,Fasteners,3024.2800,949.5182,217,31.396504
11,Machines,189238.6310,3384.7569,115,1.788618
10,Labels,12486.3120,5546.2540,364,44.418672
2,Art,27118.7920,6527.7870,796,24.071083
7,Envelopes,16476.4020,6964.1767,254,42.267582
9,Furnishings,91705.1640,13059.1436,957,14.240358
1,Appliances,107532.1610,18138.0054,466,16.867517


Lowest-profit entries: Region


,Region,Sales,Profit,Rows,Profit Margin %
0,Central,501239.8908,39706.3625,2323,7.921629
2,South,391721.9050,46749.4303,1620,11.934342
1,East,678781.2400,91522.7800,2848,13.483399
3,West,725457.8245,108418.4489,3203,14.944831


Lowest-profit entries: Category


,Category,Sales,Profit,Rows,Profit Margin %
0,Furniture,741999.7953,18451.2728,2121,2.486695
1,Office Supplies,719047.0320,122490.8008,6026,17.035158
2,Technology,836154.0330,145454.9481,1847,17.395712


In [6]:
HIGH_DISCOUNT_THRESHOLD = 0.50
if "Discount" in df.columns:
    mask = (df["Discount"] >= HIGH_DISCOUNT_THRESHOLD) & (df["Profit"] < 0)
    cols = [c for c in ["Order ID","Order Date","Customer Name","Region","Category","Sub-Category","Product Name","Sales","Profit","Discount","Ship Mode"] if c in df.columns]
    flagged = df.loc[mask, cols].sort_values(["Discount","Profit"], ascending=[False, True])
    flagged.to_csv(OUT / "high_discount_loss_orders_for_review.csv", index=False)
    print(f"High-discount (>= {HIGH_DISCOUNT_THRESHOLD:.0%}) loss-making rows: {len(flagged):,}")
    display(flagged.head(25))


High-discount (>= 50%) loss-making rows: 922


,Order ID,Order Date,Customer Name,Region,Category,Sub-Category,Product Name,Sales,Profit,Discount,Ship Mode
9774,CA-2014-169019,NaN,Luke Foster,Central,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,2177.584,-3701.8928,0.8,Standard Class
4991,US-2017-122714,2017-12-07,Henry Goldwyn,Central,Office Supplies,Binders,Ibico EPK-21 Electric Binding System,1889.990,-2929.4845,0.8,Standard Class
5310,CA-2017-131254,NaN,Nathan Cano,Central,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Bin...,1525.188,-2287.7820,0.8,First Class
1199,CA-2016-130946,2016-04-08,Zuschuss Carroll,Central,Office Supplies,Binders,GBC DocuBind P400 Electric Binding System,1088.792,-1850.9464,0.8,Standard Class
3324,CA-2014-165309,2014-11-11,Karen Daniels,Central,Office Supplies,Binders,GBC DocuBind TL300 Electric Binding System,896.990,-1480.0335,0.8,Standard Class
4820,CA-2015-140025,2015-04-07,Peter Fuller,Central,Office Supplies,Appliances,Hoover Upright Vacuum With Dirt Cup,463.248,-1181.2824,0.8,Standard Class
2846,CA-2017-152093,2017-09-10,Skye Norling,Central,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Bin...,762.594,-1143.8910,0.8,Standard Class
8640,US-2017-148551,2017-01-12,David Bremer,Central,Office Supplies,Binders,GBC Ibimaster 500 Manual ProClick Binding System,760.980,-1141.4700,0.8,Standard Class
1995,US-2017-147221,2017-12-02,Joy Smith,Central,Office Supplies,Appliances,3.6 Cubic Foot Counter Height Office Refrigerator,294.620,-766.0120,0.8,Second Class
6262,CA-2016-132479,NaN,Michael Kennedy,Central,Office Supplies,Binders,GBC ProClick 150 Presentation Binding System,442.372,-729.9138,0.8,First Class


In [7]:
sales = df["Sales"].sum()
profit = df["Profit"].sum()
kpi = pd.DataFrame({
    "Metric": ["Rows","Total Sales","Total Profit","Profit Margin %","Distinct Orders","Distinct Customers"],
    "Value": [len(df), sales, profit, profit/sales*100 if sales else np.nan,
              df["Order ID"].nunique() if "Order ID" in df.columns else np.nan,
              df["Customer ID"].nunique() if "Customer ID" in df.columns else (df["Customer Name"].nunique() if "Customer Name" in df.columns else np.nan)]
})
display(kpi)
kpi.to_csv(OUT / "kpi_reconciliation.csv", index=False)
print("Saved output files to:", OUT)


,Metric,Value
0,Rows,9.994000e+03
1,Total Sales,2.297201e+06
2,Total Profit,2.863970e+05
3,Profit Margin %,1.246722e+01
4,Distinct Orders,5.009000e+03
5,Distinct Customers,7.930000e+02


Saved output files to: A:\Data Analyst\Customer-profitability-intellegence\data\processed
